# 🧥 VTON Phase 1 — Preprocessing Pipeline

**Outputs:** parse map · pose skeleton · DensePose IUV · agnostic mask/image · cloth mask/clean

**Runtime:** T4 GPU · ~45 s for 10 images

---
## ⚠️ Required run order — read before starting

| Step | Action |
|------|--------|
| 1 | Run **Cell 1** (installs). It will auto-kill the kernel at the end. |
| 2 | Wait for Colab to say *"Your session crashed"* and reconnect (~10 s). |
| 3 | Run **Cell 2 → end** in order. **Never re-run Cell 1.** |

The auto-kill is intentional — it forces a clean Python process so numpy 1.26 loads fresh and the opencv ABI conflict never occurs.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 1 — Install ALL dependencies then auto-restart
# Run ONCE. The kernel will die at the end — that is expected.
# ════════════════════════════════════════════════════════════════════
import subprocess, os, sys

def pip(cmd):
    r = subprocess.run(f'pip install -q {cmd}', shell=True,
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f'[warn] {cmd}:\n{r.stderr[-300:]}')

# ── 1. numpy FIRST — everything else must compile against 1.26 ABI ──
# Colab ships numpy 2.x; opencv compiled for 1.x → _ARRAY_API crash.
# Must install before any other package touches numpy.
pip('numpy==1.26.4 --force-reinstall --no-deps')

# ── 2. Core image / ONNX ──
pip('opencv-python-headless==4.9.0.80')
pip('onnxruntime==1.17.3')

# ── 3. Cloth segmentation ──
pip('rembg==2.0.50')

# ── 4. Pose estimation (downloads body_pose_model on first use) ──
pip('controlnet-aux==0.0.7')

# ── 5. Diffusers stack (version depends on torch) ──
import torch
tv = tuple(int(x) for x in torch.__version__.split('.')[:2])
if tv >= (2, 3):
    pip('diffusers==0.30.3 transformers==4.40.0 accelerate==0.30.0')
else:
    pip('diffusers==0.29.2 transformers==4.39.0 accelerate==0.29.0')

# ── 6. Detectron2 + DensePose (build from source, ~3 min) ──
print('Building Detectron2 (this takes ~3 minutes)...')
pip('"git+https://github.com/facebookresearch/detectron2.git"')
pip('"git+https://github.com/facebookresearch/detectron2.git#subdirectory=projects/DensePose"')

# ── 7. Misc ──
pip('tqdm huggingface_hub')

print()
print('✅ All packages installed.')
print('Kernel will now restart automatically...')
print('When Colab says "Your session crashed", wait for reconnect then run from Cell 2.')

# Kill the kernel so numpy 1.26 loads fresh in the next session.
# If we don't do this, the already-imported numpy 2.x in memory causes
# "_ARRAY_API not found" when cv2 imports in any later cell.
os.kill(os.getpid(), 9)

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 2 — Verify environment (run after kernel restart)
# ════════════════════════════════════════════════════════════════════
import numpy as np
import cv2
import torch

np_major = int(np.__version__.split('.')[0])
assert np_major == 1, (
    f'numpy {np.__version__} detected — need 1.26.x.\n'
    'Go back and re-run Cell 1 (it will restart the kernel again).'
)

print(f'numpy  : {np.__version__}  ✅')
print(f'cv2    : {cv2.__version__}  ✅')
print(f'torch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()} | {torch.version.cuda}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print()
print('✅ Environment OK — continue to Cell 3')

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 3 — Clone project repo and set up paths
# ════════════════════════════════════════════════════════════════════
import os, sys

# ── Option A: clone your repo (replace URL when you have one) ──
# !git clone https://github.com/YOUR_USERNAME/my-vton.git

# ── Option B: create folder structure inline (used here) ──
for d in [
    'my-vton/pretrained/humanparsing',
    'my-vton/pretrained/openpose',
    'my-vton/pretrained/densepose',
    'my-vton/pretrained/u2net',
    'my-vton/preprocessing',
    'my-vton/warping',
    'my-vton/generation',
    'my-vton/fit',
    'my-vton/notebooks',
]:
    os.makedirs(d, exist_ok=True)

os.chdir('/content/my-vton')
sys.path.insert(0, '/content/my-vton')
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 4 — Download checkpoints
#
# Files and their correct source repos:
#   parsing_atr.onnx   → yisol/IDM-VTON  (Spaces repo, NOT IDM-VTON-DC)
#   parsing_lip.onnx   → yisol/IDM-VTON  (same)
#   body_pose_model    → yisol/IDM-VTON  (same)
#   densepose pkl      → Meta AI public URL
#   u2net / controlnet → auto-downloaded on first use
# ════════════════════════════════════════════════════════════════════
import os
from huggingface_hub import hf_hub_download

SPACES_REPO  = 'yisol/IDM-VTON'   # repo_type='space'
DENSEPOSE_URL = (
    'https://dl.fbaipublicfiles.com/densepose/'
    'densepose_rcnn_R_50_FPN_s1x/165712039/model_final_162be9.pkl'
)

def dl_from_space(remote_path, local_path):
    """Download one file from a HuggingFace Spaces repo."""
    if os.path.exists(local_path) and os.path.getsize(local_path) > 1_000_000:
        size_mb = os.path.getsize(local_path) / 1e6
        print(f'  [skip] {os.path.basename(local_path)} already exists ({size_mb:.0f} MB)')
        return
    print(f'  Downloading {os.path.basename(local_path)}...')
    src = hf_hub_download(
        repo_id   = SPACES_REPO,
        repo_type = 'space',
        filename  = remote_path,
        local_dir = '/tmp/hf_cache',
    )
    import shutil
    shutil.copy(src, local_path)
    size_mb = os.path.getsize(local_path) / 1e6
    print(f'  ✅ {os.path.basename(local_path)} ({size_mb:.0f} MB)')

# ── SCHP human parsing ONNX ──
print('Downloading SCHP ONNX models...')
dl_from_space('ckpt/humanparsing/parsing_atr.onnx',
              'pretrained/humanparsing/parsing_atr.onnx')
dl_from_space('ckpt/humanparsing/parsing_lip.onnx',
              'pretrained/humanparsing/parsing_lip.onnx')

# ── OpenPose checkpoint ──
print('Downloading OpenPose checkpoint...')
dl_from_space('ckpt/openpose/ckpts/body_pose_model.pth',
              'pretrained/openpose/body_pose_model.pth')

# ── DensePose checkpoint ──
print('Downloading DensePose checkpoint (~256 MB)...')
dp_path = 'pretrained/densepose/model_final_162be9.pkl'
if os.path.exists(dp_path) and os.path.getsize(dp_path) > 1_000_000:
    print(f'  [skip] already exists ({os.path.getsize(dp_path)/1e6:.0f} MB)')
else:
    os.system(f'wget -q --show-progress -O {dp_path} "{DENSEPOSE_URL}"')

# ── Summary ──
print()
print('Checkpoint verification:')
files = [
    ('pretrained/humanparsing/parsing_atr.onnx', 250),
    ('pretrained/humanparsing/parsing_lip.onnx', 250),
    ('pretrained/openpose/body_pose_model.pth',  195),
    ('pretrained/densepose/model_final_162be9.pkl', 240),
]
all_ok = True
for path, min_mb in files:
    exists = os.path.exists(path)
    size   = os.path.getsize(path) / 1e6 if exists else 0
    ok     = exists and size >= min_mb
    status = '✅' if ok else '❌ MISSING or too small'
    print(f'  {status}  {path} ({size:.0f} MB)')
    if not ok: all_ok = False

if all_ok:
    print('\n✅ All checkpoints ready — continue to Cell 5')
else:
    print('\n❌ Fix missing files before continuing')

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 5 — Write preprocessing module files
# (Skip if you cloned the repo in Cell 3 and files already exist)
# ════════════════════════════════════════════════════════════════════
from pathlib import Path

# Check if modules already exist
modules = [
    'preprocessing/human_parser.py',
    'preprocessing/pose_estimator.py',
    'preprocessing/cloth_segmentor.py',
    'preprocessing/agnostic_builder.py',
]
missing = [m for m in modules if not Path(m).exists()]

if not missing:
    print('✅ All preprocessing modules found — skipping write step')
else:
    print(f'Missing: {missing}')
    print('Please copy the preprocessing/*.py files from your repo,')
    print('or run the project setup script.')
    # If you pasted the files from the chat, they are at:
    # /content/my-vton/preprocessing/human_parser.py etc.
    # Verify they are there before proceeding.

# Create __init__.py so Python treats it as a package
Path('preprocessing/__init__.py').touch()
print('preprocessing/__init__.py: ok')

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 6 — Create synthetic test images
# (10 person + cloth pairs so the pipeline runs without VITON-HD)
# ════════════════════════════════════════════════════════════════════
import numpy as np
import cv2, os

os.makedirs('viton_hd/train/image', exist_ok=True)
os.makedirs('viton_hd/train/cloth', exist_ok=True)

pairs = []
COLORS = [
    (200,50,50),(50,200,50),(50,50,200),(200,200,50),(50,200,200),
    (200,50,200),(180,120,60),(100,150,200),(220,180,100),(80,80,80)
]

for i in range(10):
    # ── Synthetic person ──
    person = np.full((512, 384, 3), [200,170,140], dtype=np.uint8)
    cv2.ellipse(person, (192,80), (50,60), 0,0,360, (180,140,110), -1)  # head
    cv2.rectangle(person, (130,140),(254,310), (200,100,50), -1)          # torso (shirt)
    cv2.rectangle(person, (80,145), (130,280), (200,100,50), -1)         # left arm
    cv2.rectangle(person, (254,145),(310,280), (200,100,50), -1)         # right arm
    cv2.rectangle(person, (140,310),(185,480), (60,60,150),  -1)         # left leg
    cv2.rectangle(person, (200,310),(245,480), (60,60,150),  -1)         # right leg
    cv2.ellipse(person, (192,25), (30,25), 0,0,360, (30,30,30), -1)     # hair
    # add slight variation
    noise = np.random.randint(-8,8, person.shape, dtype=np.int16)
    person = np.clip(person.astype(np.int16)+noise, 0, 255).astype(np.uint8)
    cv2.imwrite(f'viton_hd/train/image/{i:04d}.jpg', person,
                [cv2.IMWRITE_JPEG_QUALITY,95])

    # ── Synthetic cloth (white bg + colored garment) ──
    cloth = np.ones((512,384,3), dtype=np.uint8)*255
    c = COLORS[i]
    cv2.rectangle(cloth, (80,60),(304,380),  c, -1)   # body
    cv2.rectangle(cloth, (20,65),(80,250),   c, -1)   # left sleeve
    cv2.rectangle(cloth, (304,65),(360,250), c, -1)   # right sleeve
    cv2.rectangle(cloth, (140,55),(244,80),  c, -1)   # collar area
    cv2.imwrite(f'viton_hd/train/cloth/{i:04d}.jpg', cloth,
                [cv2.IMWRITE_JPEG_QUALITY,95])

    pairs.append(f'{i:04d}.jpg {i:04d}.jpg')

with open('viton_hd/train_pairs.txt','w') as f:
    f.write('\n'.join(pairs))

print(f'✅ Created 10 synthetic pairs in viton_hd/train/')
print('Set USE_SYNTHETIC=False in Cell 7 if you have real VITON-HD data.')

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 7 — Load all Group A models
# ════════════════════════════════════════════════════════════════════
import sys, os
sys.path.insert(0, '/content/my-vton')

import torch
print(f'numpy  : {__import__("numpy").__version__}')
print(f'torch  : {torch.__version__}')
print(f'GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print()

from preprocessing.human_parser    import HumanParser
from preprocessing.pose_estimator  import load_pose_estimator
from preprocessing.cloth_segmentor import load_cloth_segmentor, prepare_cloth_for_training
from preprocessing.agnostic_builder import build_agnostic_pair

# A1: SCHP human parser
parser = HumanParser('pretrained/humanparsing/parsing_atr.onnx')

# A2: Pose estimator  (downloads ~400 MB weights on first run)
pose_estimator = load_pose_estimator(mode='controlnet')

# A3: Cloth segmentor  (downloads ~170 MB u2net_cloth_seg on first run)
cloth_segmentor = load_cloth_segmentor(mode='rembg')

# A4: DensePose
try:
    from preprocessing.densepose_estimator import DensePoseEstimator
    densepose = DensePoseEstimator('pretrained/densepose/model_final_162be9.pkl')
    DENSEPOSE_OK = True
    print('✅ DensePose loaded')
except Exception as e:
    print(f'⚠️  DensePose unavailable: {e}')
    print('   IUV maps will be zero arrays (pipeline still works)')
    densepose = None
    DENSEPOSE_OK = False

print('\n✅ All models loaded — continue to Cell 8')

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 8 — Run preprocessing on 10 images
# ════════════════════════════════════════════════════════════════════
import cv2, json, time, os
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

TARGET_H, TARGET_W = 512, 384
CLOTH_TYPE = 'upper'

with open('viton_hd/train_pairs.txt') as f:
    pairs = [l.strip().split() for l in f if l.strip()][:10]

results = []

for person_fname, cloth_fname in tqdm(pairs, desc='Preprocessing'):
    pid     = Path(person_fname).stem
    out_dir = Path(f'preprocessed_dataset/train/{pid}')
    out_dir.mkdir(parents=True, exist_ok=True)

    # ── Load + resize person ──
    person_bgr = cv2.imread(f'viton_hd/train/image/{person_fname}')
    person_bgr = cv2.resize(person_bgr, (TARGET_W, TARGET_H),
                            interpolation=cv2.INTER_LANCZOS4)

    # ── A1 Parse ──
    t0 = time.time()
    parse_map = parser.parse(person_bgr)
    t_parse = time.time()-t0

    # ── A2 Pose ──
    t0 = time.time()
    pose_result = pose_estimator.estimate(person_bgr)
    t_pose = time.time()-t0

    # ── A4 DensePose ──
    t0 = time.time()
    if densepose is not None:
        dp_result = densepose.estimate(person_bgr)
        iuv_img   = dp_result['iuv_img']
        iuv_color = dp_result['iuv_colored']
    else:
        iuv_img   = np.zeros((TARGET_H, TARGET_W, 3), dtype=np.uint8)
        iuv_color = iuv_img.copy()
    t_dp = time.time()-t0

    # ── Agnostic mask + image ──
    agnostic = build_agnostic_pair(
        person_bgr = person_bgr,
        parse_map  = parse_map,
        keypoints  = pose_result['keypoints'],
        cloth_type = CLOTH_TYPE,
    )

    # ── A3 Cloth segmentation ──
    cloth_bgr  = cv2.imread(f'viton_hd/train/cloth/{cloth_fname}')
    t0 = time.time()
    seg        = cloth_segmentor.segment(cloth_bgr)
    cloth_clean = prepare_cloth_for_training(
        seg['cloth_clean'], TARGET_H, TARGET_W)
    cloth_mask  = cv2.resize(seg['cloth_mask'], (TARGET_W, TARGET_H),
                             interpolation=cv2.INTER_NEAREST)
    t_cloth = time.time()-t0

    # ── Save all outputs ──
    cv2.imwrite(str(out_dir/'person.jpg'),         person_bgr, [cv2.IMWRITE_JPEG_QUALITY,95])
    cv2.imwrite(str(out_dir/'parse_map.png'),       parse_map)
    cv2.imwrite(str(out_dir/'pose_img.png'),         pose_result['pose_img'])
    cv2.imwrite(str(out_dir/'densepose_iuv.png'),    iuv_img)
    cv2.imwrite(str(out_dir/'densepose_color.png'),  iuv_color)
    cv2.imwrite(str(out_dir/'agnostic_mask.png'),    agnostic['agnostic_mask'])
    cv2.imwrite(str(out_dir/'agnostic_image.png'),   agnostic['agnostic_image'])
    cv2.imwrite(str(out_dir/'cloth_clean.png'),      cloth_clean)
    cv2.imwrite(str(out_dir/'cloth_mask.png'),       cloth_mask)
    cv2.imwrite(str(out_dir/'cloth_original.jpg'),   cloth_bgr, [cv2.IMWRITE_JPEG_QUALITY,95])
    with open(out_dir/'keypoints.json','w') as fp:
        fp.write(pose_result['keypoints_json'])

    results.append({
        'person_id':     pid,
        'cloth_id':      Path(cloth_fname).stem,
        'person':        str(out_dir/'person.jpg'),
        'parse_map':     str(out_dir/'parse_map.png'),
        'pose_img':      str(out_dir/'pose_img.png'),
        'keypoints_json':str(out_dir/'keypoints.json'),
        'densepose_iuv': str(out_dir/'densepose_iuv.png'),
        'agnostic_mask': str(out_dir/'agnostic_mask.png'),
        'agnostic_image':str(out_dir/'agnostic_image.png'),
        'cloth_clean':   str(out_dir/'cloth_clean.png'),
        'cloth_mask':    str(out_dir/'cloth_mask.png'),
        'timing': {'parse':t_parse,'pose':t_pose,'densepose':t_dp,'cloth':t_cloth},
    })

os.makedirs('preprocessed_dataset', exist_ok=True)
with open('preprocessed_dataset/train_manifest.json','w') as f:
    json.dump(results, f, indent=2)

print(f'\n✅ Processed {len(results)} samples')
t = results[-1]['timing']
for k,v in t.items():
    print(f'  {k:12s}: {v*1000:.0f} ms')

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 9 — Visualize outputs for one sample
# ════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import cv2, numpy as np
from preprocessing.human_parser import HumanParser, ATR_LABELS

SAMPLE_IDX = 0
rec = results[SAMPLE_IDX]

def load_rgb(p):  return cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
def load_gray(p): return cv2.imread(p, cv2.IMREAD_GRAYSCALE)

parse_map   = load_gray(rec['parse_map'])
parse_color = cv2.cvtColor(HumanParser.colorize(parse_map), cv2.COLOR_BGR2RGB)

panels = [
    (load_rgb(rec['person']),        'Person',          None),
    (parse_color,                    'Parse map',       None),
    (load_rgb(rec['pose_img']),      'Pose skeleton',   None),
    (load_rgb(rec['densepose_iuv']), 'DensePose IUV',   None),
    (load_rgb(rec['agnostic_image']),'Agnostic image',  None),
    (load_gray(rec['agnostic_mask']),'Agnostic mask',   'gray'),
    (load_rgb(rec['cloth_clean']),   'Cloth clean',     None),
    (load_gray(rec['cloth_mask']),   'Cloth mask',      'gray'),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle(f'Phase 1 outputs — sample {rec["person_id"]}',
             fontsize=14, fontweight='bold')
for ax, (img, title, cmap) in zip(axes.ravel(), panels):
    ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.savefig('preprocessed_dataset/sample_vis.png', dpi=150, bbox_inches='tight')
plt.show()

print('Parse labels present:')
for lbl in np.unique(parse_map):
    name = ATR_LABELS[lbl] if lbl < len(ATR_LABELS) else f'Unknown-{lbl}'
    pct  = (parse_map==lbl).mean()*100
    print(f'  [{lbl:2d}] {name:20s} {pct:.1f}%')

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 10 — All 10 samples grid
# ════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import cv2

fig, axes = plt.subplots(len(results), 4, figsize=(14, len(results)*3.5))
col_titles = ['Person', 'Agnostic Image', 'Agnostic Mask', 'Cloth Clean']

for j, t in enumerate(col_titles):
    axes[0,j].set_title(t, fontsize=10, fontweight='bold')

for i, rec in enumerate(results):
    imgs = [
        cv2.cvtColor(cv2.imread(rec['person']),         cv2.COLOR_BGR2RGB),
        cv2.cvtColor(cv2.imread(rec['agnostic_image']), cv2.COLOR_BGR2RGB),
        cv2.imread(rec['agnostic_mask'], cv2.IMREAD_GRAYSCALE),
        cv2.cvtColor(cv2.imread(rec['cloth_clean']),    cv2.COLOR_BGR2RGB),
    ]
    cmaps = [None, None, 'gray', None]
    for j,(img,cmap) in enumerate(zip(imgs,cmaps)):
        axes[i,j].imshow(img, cmap=cmap)
        axes[i,j].axis('off')
    axes[i,0].set_ylabel(rec['person_id'], fontsize=7, rotation=0, ha='right')

plt.suptitle('Phase 1 — All samples', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('preprocessed_dataset/all_samples_grid.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 11 — Final checklist
# ════════════════════════════════════════════════════════════════════
import os, glob
from pathlib import Path

if torch.cuda.is_available():
    r = torch.cuda.memory_reserved()/1e9
    t = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f'GPU memory: {r:.1f} / {t:.1f} GB used\n')

print('='*50)
print('PHASE 1 CHECKLIST')
print('='*50)

checks = [
    ('pretrained/humanparsing/parsing_atr.onnx',    'SCHP ONNX checkpoint'),
    ('pretrained/densepose/model_final_162be9.pkl', 'DensePose checkpoint'),
    ('preprocessed_dataset/train_manifest.json',    'Dataset manifest'),
    ('preprocessed_dataset/sample_vis.png',         'Single-sample visualization'),
    ('preprocessed_dataset/all_samples_grid.png',   'All-samples grid'),
]

all_ok = True
for path, desc in checks:
    found = Path(path).exists()
    print(f'  {"✅" if found else "❌"}  {desc}')
    if not found: all_ok = False

print()
if all_ok:
    print('🎉 Phase 1 complete! Ready for Phase 2 — Warping Module.')
else:
    print('⚠️  Some checks failed — see errors above.')